# Lab 5 — Ensemble Detector (Random Forest)

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Learning objectives

1. Train a **Random Forest** ensemble with **100** trees.
2. Use `class_weight='balanced'` to handle fraud imbalance.
3. Report **precision**, **recall**, and **F1** on the fraud class.
4. Compare RF to LOF (Lab 4) and resampled logistic (Lab 3).

> **Checkpoints:** precision **1.00** · recall **0.50** · F1 ≈ **0.67**

**Companion script:** `../scripts/lab05_ensemble_detector.py`

## Random Forest for fraud

| Idea | Benefit on tabular fraud data |
|------|------------------------------|
| **Bagging** | Many trees reduce variance |
| **Mixed features** | Handles numeric + one-hot `merchant_category` |
| **class_weight** | Up-weights rare fraud without explicit oversampling |

RF trades off vs LOF: **higher precision**, **lower recall** on this test split.

---

## 1. Load data and split

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-06":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "credit-card" / "credit_card_transactions.csv").is_file():
            GH_ROOT = parent
            break

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"test:  {len(X_test)} (fraud {int(y_test.sum())})")

---

## 2. Train Random Forest pipeline

In [ ]:
N_ESTIMATORS = 100

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=N_ESTIMATORS,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("Lab 5 — Ensemble detector (Random Forest)")
print(f"estimators: {N_ESTIMATORS}, class_weight: balanced")

---

## 3. Fraud-class metrics

In [ ]:
f1 = f1_score(y_test, y_pred, zero_division=0)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"precision (fraud): {precision:.4f}")
print(f"recall (fraud): {recall:.4f}")
print(f"F1 (fraud): {f1:.4f}")

metrics = pd.DataFrame({
    "metric": ["precision", "recall", "F1"],
    "fraud_class": [precision, recall, f1],
})
display(metrics.round(4))

Perfect **precision** (1.0) — no false fraud alarms on test. **Recall** 0.5 — catches 1 of 2 test frauds.

---

## 4. Predictions on test set

In [ ]:
pred_df = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "amount": X_test["amount"].values,
    "distance_from_home": X_test["distance_from_home"].values,
})
display(pred_df.sort_values("predicted", ascending=False).head(8).round(2))

---

## 5. Compare to Labs 3–4

In [ ]:
compare = pd.DataFrame({
    "model": ["Logistic + oversample (Lab 3)", "LOF (Lab 4)", "Random Forest (Lab 5)"],
    "precision": ["varies", 0.3333, precision],
    "recall": [0.6667, 1.0, recall],
    "F1": [0.6667, "—", f1],
})
display(compare)

---

## 6. Extension — n_estimators = 200

In [ ]:
pipe_200 = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)
pipe_200.fit(X_train, y_train)
f1_200 = f1_score(y_test, pipe_200.predict(X_test), zero_division=0)

display(pd.DataFrame({
    "n_estimators": [100, 200],
    "F1_fraud": [f1, f1_200],
}).round(4))

---

## 7. Feature importance (optional)

In [ ]:
feature_names = (
    NUMERIC_FEATURES
    + list(pipe.named_steps["preprocess"].named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
)
importances = pipe.named_steps["clf"].feature_importances_
imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).head(8)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=imp_df, x="importance", y="feature", ax=ax, palette="Blues_r")
ax.set_title("Random Forest — top feature importances")
plt.tight_layout()
plt.show()

---

## 8. Checkpoint summary

In [ ]:
assert N_ESTIMATORS == 100
assert abs(precision - 1.0) < 0.01
assert abs(recall - 0.5) < 0.01
assert abs(f1 - 0.6667) < 0.05
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why might a bank prefer RF over LOF for auto-decline rules?
2. What does `class_weight='balanced'` do vs oversampling (Lab 3)?
3. Which model will Lab 6 rank best overall?

**Previous:** [Lab 4 — Proximity detector](lab04_proximity_detector.ipynb)  
**Next:** [Lab 6 — Capstone fraud report](lab06_capstone_fraud_report.ipynb)